In [5]:
import xgboost
print(xgboost.__version__)

3.2.0


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv("../data/neurosense_cleaned.csv")

print("Dataset shape:", df.shape)

# =====================================================
# FEATURES / LABELS
# =====================================================

drop_cols = ["label", "emotion"]

X = df.drop(columns=[col for col in drop_cols if col in df.columns])

meta_cols = ["subject", "session", "trial", "sample"]

for col in meta_cols:
    if col in X.columns:
        X = X.drop(columns=col)

y = df["label"]

groups = df["subject"]

print("Features:", X.shape[1])
print("Classes:", np.unique(y))

# =====================================================
# GROUP SPLIT
# =====================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

# =====================================================
# MODEL
# =====================================================

xgb = XGBClassifier(
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

# =====================================================
# HYPERPARAMETER SEARCH
# =====================================================

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=10,
    cv=2,
    scoring="f1_weighted",
    random_state=42,
    verbose=2,
    n_jobs=-1
)

print("\nTraining XGBoost...")
search.fit(X_train, y_train)

best_model = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)

# =====================================================
# PREDICTIONS
# =====================================================

y_pred = best_model.predict(X_test)

# =====================================================
# METRICS
# =====================================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("\n" + "=" * 60)
print("XGBOOST SUBJECT-DEPENDENT RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =====================================================
# CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# =====================================================
# SAVE RESULTS
# =====================================================

results = pd.DataFrame([
    {
        "Model": "XGBoost Subject-Dependent",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Best Parameters": str(search.best_params_)
    }
])

results.to_csv(
    "results_xgboost_subject_dependent.csv",
    index=False
)

print("\nResults saved successfully.")

Dataset shape: (37575, 361)
Features: 355
Classes: [0 1 2 3]

Train shape: (30060, 355)
Test shape: (7515, 355)

Training XGBoost...
Fitting 2 folds for each of 10 candidates, totalling 20 fits

Best Parameters:
{'subsample': 0.8, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

XGBOOST SUBJECT-DEPENDENT RESULTS
Accuracy : 0.3860
Precision: 0.4262
Recall   : 0.3860
F1-score : 0.3820

Classification Report:

              precision    recall  f1-score   support

           0       0.41      0.49      0.45      1950
           1       0.30      0.49      0.37      1905
           2       0.46      0.32      0.38      1947
           3       0.55      0.23      0.33      1713

    accuracy                           0.39      7515
   macro avg       0.43      0.38      0.38      7515
weighted avg       0.43      0.39      0.38      7515


Confusion Matrix:
[[947 639 139 225]
 [463 925 428  89]
 [336 964 630  17]
 [545 598 171 399]]

Results saved succes